# Solution Evaluation Dashboard

Visualise a single `output_payload` JSON solution.

- **Gantt chart** — driver timelines (free time / driver move / service labor)
- **Distance per service** — stacked bar (labor + move)
- **Distance per driver** — stacked bar (labor + move)
- **Summary metrics** — from an optional evaluation report

All logic lives in `src/analysis/solution_evaluation.py`. Edit the **Configuration** cell only.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
from typing import Optional

_PROJECT_ROOT = Path("..").resolve()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from src.analysis.solution_evaluation import (
    load_payload,
    filter_by_date,
    flatten_labors,
    reconstruct_timeline,
    build_gantt_figure,
    build_service_distance_figure,
    build_driver_distance_figure,
    build_summary_table,
    build_route_map,
)
from src.optimization.settings.model_params import ModelParams

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────────────

SOL_PAYLOAD    = Path("../data/runs/run-api-f0d19f6b/output/output_payload.json")
EVAL: Optional[Path] = None  # e.g. Path("../data/runs/run-X/diagnostics/solution_evaluation_report.json")
LABEL          = "solution"
PLANNING_DATE: Optional[str] = None  # e.g. "2026-02-18"

# ── Route map ───────────────────────────────────────────────────────────────────────────
DRIVER_DIRECTORY: Optional[Path] = None  # e.g. Path("../data/model_input/driver_directory.json")

_params = ModelParams()
ALFRED_SPEED_KMH: float = _params.alfred_speed_kmh
print(f"alfred_speed_kmh={ALFRED_SPEED_KMH}")

In [3]:
# ── Load and process ───────────────────────────────────────────────────────────────────
services = load_payload(SOL_PAYLOAD)
if PLANNING_DATE:
    services = filter_by_date(services, PLANNING_DATE)

rows, _ = flatten_labors(services)
segments = reconstruct_timeline(rows, ALFRED_SPEED_KMH)

drivers      = sorted({r["driver_id"]      for r in rows if r["driver_id"]  is not None})
all_services = sorted({str(r["service_id"]) for r in rows if r["service_id"] is not None})

print(f"{LABEL}: {len(rows)} labors | {len(drivers)} drivers | {len(segments)} segments")

solution: 66 labors | 13 drivers | 155 segments


---
## Gantt Chart — Driver Timelines

In [4]:
build_gantt_figure(segments, drivers, LABEL).show()

---
## Distance per Service

In [5]:
build_service_distance_figure(rows, all_services, LABEL).show()

---
## Distance per Driver

In [6]:
build_driver_distance_figure(rows, drivers, LABEL).show()

---
## Summary Metrics
_Requires `EVAL` to be set._

In [ ]:
tbl = build_summary_table(EVAL, LABEL)
if tbl is not None:
    display(tbl)

---
## Route Map

Select a driver from the dropdown to visualise their geographic route.

Segment colours:
- **Blue** — service labor (vehicle transportation)
- **Orange** — driver move (inter-service travel)
- Dashed lines indicate straight-line fallback when OSRM is unavailable.

In [ ]:
import json as _json
import ipywidgets as widgets
from IPython.display import display


def _resolve_home_wkt(driver_id: str):
    if not DRIVER_DIRECTORY or not DRIVER_DIRECTORY.exists():
        return None
    from src.data.parsing.driver_directory_parser import DriverDirectoryParser
    _dir_df = DriverDirectoryParser.parse(
        _json.loads(DRIVER_DIRECTORY.read_text(encoding="utf-8"))
    )
    if _dir_df.empty:
        return None
    row = _dir_df[_dir_df["driver_id"].astype(str) == driver_id]
    if row.empty:
        return None
    r = row.iloc[0]
    return f"POINT({r['longitud']} {r['latitud']})"


_drv_dropdown = widgets.Dropdown(
    options=drivers,
    description="Driver:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px"),
)
_map_output = widgets.Output()


def _on_driver_change(change):
    _map_output.clear_output(wait=True)
    with _map_output:
        try:
            m = build_route_map(
                services=services,
                rows=rows,
                driver_id=change["new"],
                driver_home_wkt=_resolve_home_wkt(change["new"]),
                label=LABEL,
            )
            display(m)
        except Exception as e:
            print(f"Could not render route for driver {change['new']!r}: {e}")


_drv_dropdown.observe(_on_driver_change, names="value")
_on_driver_change({"new": _drv_dropdown.value})
display(_drv_dropdown, _map_output)